In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()

if not (NOTEBOOK_DIR / "_bootstrap.py").exists():
    candidates = [NOTEBOOK_DIR, NOTEBOOK_DIR.parent, NOTEBOOK_DIR.parent.parent]
    for candidate in candidates:
        if (candidate / "_bootstrap.py").exists():
            NOTEBOOK_DIR = candidate
            break

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _bootstrap import PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, RESULTS_DIR

print(PROJECT_ROOT)
print(RAW_DIR)
print(PROCESSED_DIR)
print(OUTPUT_DIR)
print(RESULTS_DIR)

/Users/rodrigue.lawson/VSCode Projects/lexcare-ai
/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/raw
/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/processed
/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/experiments/output
/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/experiments/results


In [2]:
from app.repositories.source_registry import SourceRegistry
from app.services.hub_source_service import HubSourceService

registry = SourceRegistry(registry_path=str(PROJECT_ROOT / "data" / "source_registry.yaml"))
service = HubSourceService(source_registry=registry)

created = service.sync_sources()
print(created)

[]


In [3]:
hub_sources = service.repository.load_all()
hub_sources

[HubSource(source_key='078626d6-1694-5e6c-8ca8-29d1f6415cb4', source_id='gesetze-im-internet', source_name='Gesetze im Internet', created_at=datetime.datetime(2026, 6, 28, 8, 13, 17, 322435, tzinfo=datetime.timezone.utc)),
 HubSource(source_key='b0629086-d7e9-5724-9279-dd5d46011a87', source_id='gba', source_name='Gemeinsamer Bundesausschuss', created_at=datetime.datetime(2026, 6, 28, 8, 13, 17, 323346, tzinfo=datetime.timezone.utc)),
 HubSource(source_key='60e027c9-bc85-5314-b7ce-e1dd5c8498d2', source_id='bundestag', source_name='Bundestag Open Data', created_at=datetime.datetime(2026, 6, 28, 8, 13, 17, 323852, tzinfo=datetime.timezone.utc)),
 HubSource(source_key='13aa6204-359f-52f1-a228-fe22f04734c7', source_id='bmg-web', source_name='Bundesministerium für Gesundheit', created_at=datetime.datetime(2026, 6, 28, 8, 13, 17, 324184, tzinfo=datetime.timezone.utc))]

In [4]:
import pandas as pd

hub_sources_df = pd.DataFrame([
    {
        "source_key": item.source_key,
        "source_id": item.source_id,
        "source_name": item.source_name,
        "created_at": item.created_at,
    }
    for item in hub_sources
])

hub_sources_df

,source_key,source_id,source_name,created_at
0,078626d6-1694-5e6c-8ca8-29d1f6415cb4,gesetze-im-internet,Gesetze im Internet,2026-06-28 08:13:17.322435+00:00
1,b0629086-d7e9-5724-9279-dd5d46011a87,gba,Gemeinsamer Bundesausschuss,2026-06-28 08:13:17.323346+00:00
2,60e027c9-bc85-5314-b7ce-e1dd5c8498d2,bundestag,Bundestag Open Data,2026-06-28 08:13:17.323852+00:00
3,13aa6204-359f-52f1-a228-fe22f04734c7,bmg-web,Bundesministerium für Gesundheit,2026-06-28 08:13:17.324184+00:00


In [5]:
from app.repositories.source_registry import SourceRegistry
from app.repositories.document_repository import FileDocumentRepository
from app.services.incremental_ingestion_service import IncrementalIngestionService

registry = SourceRegistry(registry_path=str(PROJECT_ROOT / "data" / "source_registry.yaml"))
document_repo = FileDocumentRepository()

ingestion_service = IncrementalIngestionService(
    source_registry=registry,
    document_repository=document_repo,
)

summary = ingestion_service.ingest_source("gesetze-im-internet")
summary

IngestionSummary(source_id='gesetze-im-internet', fetched=3, ingested=3, skipped=0, updated=0, errors=0)

In [6]:
from app.repositories.document_repository import FileDocumentRepository
from app.repositories.hub_source_repository import FileHubSourceRepository
from app.repositories.hub_document_repository import FileHubDocumentRepository
from app.services.hub_document_service import HubDocumentService

document_repo = FileDocumentRepository()
source_repo = FileHubSourceRepository()
hub_doc_repo = FileHubDocumentRepository()

doc_service = HubDocumentService(
    document_repository=document_repo,
    hub_source_repository=source_repo,
    hub_document_repository=hub_doc_repo,
)

created_docs = doc_service.sync_documents()
print(created_docs)

[HubDocument(document_key='105f3cb3-a3d5-5bc8-90fb-853b86735460', document_id='gesetze_im_internet::pueg_refe_pflegereform_bf_pdf', source_key='078626d6-1694-5e6c-8ca8-29d1f6415cb4', source_id='gesetze-im-internet', document_name='PUEG_RefE_Pflegereform_bf.pdf', source_path='file:///Users/rodrigue.lawson/VSCode%20Projects/lexcare-ai/data/raw/gesetze_im_internet/PUEG_RefE_Pflegereform_bf.pdf', created_at=datetime.datetime(2026, 6, 28, 10, 30, 16, 308371, tzinfo=datetime.timezone.utc)), HubDocument(document_key='b61bc03c-6064-53bc-8c53-723fc86d1e51', document_id='gesetze_im_internet::sgb_5_pdf', source_key='078626d6-1694-5e6c-8ca8-29d1f6415cb4', source_id='gesetze-im-internet', document_name='SGB_5.pdf', source_path='file:///Users/rodrigue.lawson/VSCode%20Projects/lexcare-ai/data/raw/gesetze_im_internet/SGB_5.pdf', created_at=datetime.datetime(2026, 6, 28, 10, 30, 16, 308823, tzinfo=datetime.timezone.utc)), HubDocument(document_key='7956037b-e679-5bfa-ae40-95c491bdb1cc', document_id='ges

In [7]:
hub_documents = hub_doc_repo.load_all()
hub_documents

[HubDocument(document_key='105f3cb3-a3d5-5bc8-90fb-853b86735460', document_id='gesetze_im_internet::pueg_refe_pflegereform_bf_pdf', source_key='078626d6-1694-5e6c-8ca8-29d1f6415cb4', source_id='gesetze-im-internet', document_name='PUEG_RefE_Pflegereform_bf.pdf', source_path='file:///Users/rodrigue.lawson/VSCode%20Projects/lexcare-ai/data/raw/gesetze_im_internet/PUEG_RefE_Pflegereform_bf.pdf', created_at=datetime.datetime(2026, 6, 28, 10, 30, 16, 308371, tzinfo=datetime.timezone.utc)),
 HubDocument(document_key='b61bc03c-6064-53bc-8c53-723fc86d1e51', document_id='gesetze_im_internet::sgb_5_pdf', source_key='078626d6-1694-5e6c-8ca8-29d1f6415cb4', source_id='gesetze-im-internet', document_name='SGB_5.pdf', source_path='file:///Users/rodrigue.lawson/VSCode%20Projects/lexcare-ai/data/raw/gesetze_im_internet/SGB_5.pdf', created_at=datetime.datetime(2026, 6, 28, 10, 30, 16, 308823, tzinfo=datetime.timezone.utc)),
 HubDocument(document_key='7956037b-e679-5bfa-ae40-95c491bdb1cc', document_id='g

In [8]:
from app.repositories.hub_topic_repository import FileHubTopicRepository
from app.services.hub_topic_service import HubTopicService

topic_repo = FileHubTopicRepository()

topic_service = HubTopicService(
    source_registry=registry,
    repository=topic_repo,
)

created_topics = topic_service.sync_topics()
print(created_topics)

[]


In [9]:
hub_topics = topic_repo.load_all()
hub_topics

[HubTopic(topic_key='ae4ec098-55d5-5abe-8993-5d03bb12e9d1', topic_id='healthcare_law', topic_name='healthcare-law', created_at=datetime.datetime(2026, 6, 28, 8, 13, 29, 573749, tzinfo=datetime.timezone.utc)),
 HubTopic(topic_key='fd5a60ce-d79b-5a08-923d-02b7404dfc4f', topic_id='healthcare_regulation', topic_name='healthcare-regulation', created_at=datetime.datetime(2026, 6, 28, 8, 13, 29, 574136, tzinfo=datetime.timezone.utc)),
 HubTopic(topic_key='2161c68e-8d10-51f2-b6d7-bf5de7efd556', topic_id='legislation', topic_name='legislation', created_at=datetime.datetime(2026, 6, 28, 8, 13, 29, 574362, tzinfo=datetime.timezone.utc)),
 HubTopic(topic_key='cc13fcf4-77dd-513d-adc0-499c30eac4d7', topic_id='healthcare_policy', topic_name='healthcare-policy', created_at=datetime.datetime(2026, 6, 28, 8, 13, 29, 574558, tzinfo=datetime.timezone.utc))]

In [10]:
import pandas as pd

hub_topics_df = pd.DataFrame([
    {
        "topic_key": item.topic_key,
        "topic_id": item.topic_id,
        "topic_name": item.topic_name,
        "created_at": item.created_at,
    }
    for item in hub_topics
])

hub_topics_df

,topic_key,topic_id,topic_name,created_at
0,ae4ec098-55d5-5abe-8993-5d03bb12e9d1,healthcare_law,healthcare-law,2026-06-28 08:13:29.573749+00:00
1,fd5a60ce-d79b-5a08-923d-02b7404dfc4f,healthcare_regulation,healthcare-regulation,2026-06-28 08:13:29.574136+00:00
2,2161c68e-8d10-51f2-b6d7-bf5de7efd556,legislation,legislation,2026-06-28 08:13:29.574362+00:00
3,cc13fcf4-77dd-513d-adc0-499c30eac4d7,healthcare_policy,healthcare-policy,2026-06-28 08:13:29.574558+00:00


In [11]:
from app.repositories.link_document_topic_repository import FileLinkDocumentTopicRepository
from app.services.link_document_topic_service import LinkDocumentTopicService

link_repo = FileLinkDocumentTopicRepository()

link_service = LinkDocumentTopicService(
    document_repository=document_repo,
    hub_document_repository=hub_doc_repo,
    hub_topic_repository=topic_repo,
    link_repository=link_repo,
)

created_links = link_service.sync_links()
print(created_links)

[LinkDocumentTopic(link_key='a7b3a221-9d02-5e2d-ba4f-10b7d73ad63d', document_key='105f3cb3-a3d5-5bc8-90fb-853b86735460', topic_key='ae4ec098-55d5-5abe-8993-5d03bb12e9d1', created_at=datetime.datetime(2026, 6, 28, 10, 30, 16, 361125, tzinfo=datetime.timezone.utc)), LinkDocumentTopic(link_key='c6551f81-e385-545a-a187-730331d87133', document_key='b61bc03c-6064-53bc-8c53-723fc86d1e51', topic_key='ae4ec098-55d5-5abe-8993-5d03bb12e9d1', created_at=datetime.datetime(2026, 6, 28, 10, 30, 16, 361670, tzinfo=datetime.timezone.utc)), LinkDocumentTopic(link_key='b20e4897-08d1-57d4-91e3-08b11a89bf89', document_key='7956037b-e679-5bfa-ae40-95c491bdb1cc', topic_key='ae4ec098-55d5-5abe-8993-5d03bb12e9d1', created_at=datetime.datetime(2026, 6, 28, 10, 30, 16, 362187, tzinfo=datetime.timezone.utc))]


In [12]:
link_repo.load_all()

[LinkDocumentTopic(link_key='a7b3a221-9d02-5e2d-ba4f-10b7d73ad63d', document_key='105f3cb3-a3d5-5bc8-90fb-853b86735460', topic_key='ae4ec098-55d5-5abe-8993-5d03bb12e9d1', created_at=datetime.datetime(2026, 6, 28, 10, 30, 16, 361125, tzinfo=datetime.timezone.utc)),
 LinkDocumentTopic(link_key='c6551f81-e385-545a-a187-730331d87133', document_key='b61bc03c-6064-53bc-8c53-723fc86d1e51', topic_key='ae4ec098-55d5-5abe-8993-5d03bb12e9d1', created_at=datetime.datetime(2026, 6, 28, 10, 30, 16, 361670, tzinfo=datetime.timezone.utc)),
 LinkDocumentTopic(link_key='b20e4897-08d1-57d4-91e3-08b11a89bf89', document_key='7956037b-e679-5bfa-ae40-95c491bdb1cc', topic_key='ae4ec098-55d5-5abe-8993-5d03bb12e9d1', created_at=datetime.datetime(2026, 6, 28, 10, 30, 16, 362187, tzinfo=datetime.timezone.utc))]

In [13]:
import pandas as pd

links_df = pd.DataFrame([
    {
        "link_key": item.link_key,
        "document_key": item.document_key,
        "topic_key": item.topic_key,
        "created_at": item.created_at,
    }
    for item in link_repo.load_all()
])

links_df

,link_key,document_key,topic_key,created_at
0,a7b3a221-9d02-5e2d-ba4f-10b7d73ad63d,105f3cb3-a3d5-5bc8-90fb-853b86735460,ae4ec098-55d5-5abe-8993-5d03bb12e9d1,2026-06-28 10:30:16.361125+00:00
1,c6551f81-e385-545a-a187-730331d87133,b61bc03c-6064-53bc-8c53-723fc86d1e51,ae4ec098-55d5-5abe-8993-5d03bb12e9d1,2026-06-28 10:30:16.361670+00:00
2,b20e4897-08d1-57d4-91e3-08b11a89bf89,7956037b-e679-5bfa-ae40-95c491bdb1cc,ae4ec098-55d5-5abe-8993-5d03bb12e9d1,2026-06-28 10:30:16.362187+00:00
